In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io
from helper import *

from sklearn.model_selection import GridSearchCV

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
!pip install mlflow albumentations -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import mlflow
import mlflow.pytorch

In [5]:
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils

In [6]:
import os

if not os.path.exists("skin-dataset-classification"):
    !git clone https://github.com/2245-RN-ITBA/skin-dataset-classification.git

os.chdir("skin-dataset-classification")
print(os.getcwd())
print(os.listdir("data/Split_smol"))

Cloning into 'skin-dataset-classification'...
remote: Enumerating objects: 934, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 934 (delta 17), reused 15 (delta 15), pack-reused 911 (from 2)
Receiving objects: 100% (934/934), 222.43 MiB | 19.37 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/skin-dataset-classification
['val', 'train']


In [7]:
mlflow.set_tracking_uri("sqlite:///mlflow_3.db")
mlflow.set_experiment("Clasificador_Imagenes")

2026/06/22 11:51:44 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/22 11:51:44 INFO mlflow.store.db.utils: Updating database tables
2026/06/22 11:51:47 INFO mlflow.tracking.fluent: Experiment with name 'Clasificador_Imagenes' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/skin-dataset-classification/mlruns/1', creation_time=1782129107324, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782129107324, lifecycle_stage='active', name='Clasificador_Imagenes', tags={}, trace_location=None, workspace='default'>

## Funciones auxiliares

Se definen acá para que el notebook sea autocontenido (en el original venían de `helper.py`).

In [8]:
# NUEVA

def plot_to_tensorboard(fig, writer, tag, step):
    """Convierte una figura matplotlib a imagen y la loguea en TensorBoard."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    image = Image.open(buf).convert("RGB")
    image = np.array(image)
    image = torch.tensor(image).permute(2, 0, 1) / 255.0
    writer.add_image(tag, image, global_step=step)
    plt.close(fig)


def count_parameters(model):
    """Cuenta los parámetros entrenables del modelo."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [9]:
# MODIFICADA

def log_classification_report(model, loader, writer, device, classes, step, prefix="val"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Matriz de confusion
    cm = confusion_matrix(all_labels, all_preds)
    fig_cm, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{prefix.title()} - Confusion Matrix')

    # Guardar localmente y subir a MLflow
    fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
    fig_cm.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    os.remove(fig_path)

    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step)

    cls_report = classification_report(all_labels, all_preds, target_names=classes,
                                       zero_division=0)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

    # También loguear texto del reporte
    with open(f"classification_report_{prefix}_epoch_{step}.txt", "w") as f:
        f.write(cls_report)
    mlflow.log_artifact(f.name)
    os.remove(f.name)

In [10]:
def evaluate(model, loader, writer, device, classes, criterion, epoch=None, prefix="val"):
    log_classification_report(model, loader, writer, device, classes, step=epoch, prefix=prefix)
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss_sum += loss.item()
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)

            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    acc      = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss",     avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc,      epoch)

    return avg_loss, acc

## Dataset

In [11]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir  = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels      = []

        class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

        for cls in class_names:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(cls_dir, fname))
                    self.labels.append(cls)

        self.label_encoder = LabelEncoder()
        self.labels = self.label_encoder.fit_transform(self.labels)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        label = self.labels[idx]
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]
        return image, label

## Arquitectura CNN

La `CNNClassifier` sigue el patrón clásico de una CNN para clasificación:

```
Entrada (3, H, W)
  → Bloque conv 1: Conv2d(3→32)  + BatchNorm + ReLU + MaxPool  → (32, H/2, W/2)
  → Bloque conv 2: Conv2d(32→64) + BatchNorm + ReLU + MaxPool  → (64, H/4, W/4)
  → Bloque conv 3: Conv2d(64→128)+ BatchNorm + ReLU + MaxPool  → (128, H/8, W/8)
  → AdaptiveAvgPool2d(4,4)                                      → (128, 4, 4)
  → Flatten                                                     → (2048,)
  → Linear(2048 → 256) + ReLU + Dropout
  → Linear(256 → num_classes)
```

**Decisiones de diseño:**
- `BatchNorm2d` después de cada convolución: estabiliza el entrenamiento y actúa como regularizador.
- `MaxPool2d(2,2)`: reduce la resolución espacial a la mitad, agrandando el campo receptivo.
- `AdaptiveAvgPool2d(4,4)`: hace que el clasificador funcione con cualquier `input_size` (32, 64, 128) sin cambiar la arquitectura.
- `Dropout` solo en la parte densa (clasificador), no en las convoluciones.
- Inicialización **He (Kaiming)** para los filtros convolucionales y las capas lineales, ya que se usa ReLU.

In [12]:
class CNNClassifier(nn.Module):
    """
    CNN con 3 bloques convolucionales + clasificador denso.

    Parámetros
    ----------
    num_classes : int   - número de clases de salida
    input_size  : int   - resolución de la imagen de entrada (se usa solo para documentación;
                          AdaptiveAvgPool hace que la red sea agnóstica al tamaño)
    dropout     : float - probabilidad de Dropout en la capa densa (0.0 = sin Dropout)
    """
    def __init__(self, num_classes=10, input_size=64, dropout=0.0):
        super().__init__()

        # Bloque conv 1: 3 → 32
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Bloque conv 2: 32 → 64
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Bloque conv 3: 64 → 128
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Pooling adaptativo: salida siempre (128, 4, 4) sin importar el input_size
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        # Clasificador denso
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(256, num_classes)
        )

        # Inicialización He para Conv2d y Linear
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.adaptive_pool(x)
        x = self.classifier(x)
        return x

In [13]:
# Verificación rápida: probar que la red acepta distintos input_size
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for sz in [32, 64, 128]:
    dummy = torch.randn(2, 3, sz, sz).to(device)
    m     = CNNClassifier(num_classes=9, input_size=sz, dropout=0.3).to(device)
    out   = m(dummy)
    print(f"input_size={sz:3d} → output shape: {out.shape}  params: {count_parameters(m):,}")

input_size= 32 → output shape: torch.Size([2, 9])  params: 620,553
input_size= 64 → output shape: torch.Size([2, 9])  params: 620,553
input_size=128 → output shape: torch.Size([2, 9])  params: 620,553


## Espacio de búsqueda de hiperparámetros

Mismo espacio que el Notebook 2 (MLP) para poder comparar resultados directamente.

| HP | Valores |
|---|---|
| `input_size` | 32, 64, 128 |
| `batch_size` | 16, 64, 128 |
| `lr` | 1e-2, 1e-3, 1e-4 |
| `optimizer` | Adam, SGD |
| `HFlip` | 0.0, 0.5 |
| `VFlip` | 0.0, 0.5 |
| `RBContrast` | 0.0, 0.5 |
| `dropout` | 0.0, 0.1, 0.2, 0.3 |

Total de combinaciones: `3×3×3×2×2×2×2×4 = 1728`. Se muestrea ~5% aleatoriamente.

In [14]:
# Paths
train_dir = "data/Split_smol/train"
val_dir = "data/Split_smol/val/"


In [15]:
# Crear directorio de logs de tensorboard
log_dir = "runs/experimento_skin"
writer = SummaryWriter(log_dir=log_dir)

In [16]:
np.random.rand()


0.13655073015757857

In [17]:
# reduje los hiperparametros para que corra
hparams_space = {
    "model":       "CNNClassifier",
    "input_size":  [32, 64],
    "batch_size":  [16, 64],
    "lr":          [1e-2, 1e-3, 1e-4],
    "epochs":      50,
    "optimizer":   ["Adam", "SGD"],
    "HFlip":       [0.0, 0.5],
    "VFlip":       [0.0, 0.5],
    "RBContrast":  [0.0, 0.5],
    "loss_fn":     "CrossEntropyLoss",
    "train_dir":   train_dir,
    "val_dir":     val_dir,
    "es_patience": 5,
    "dropout":     [0.0, 0.2],
}


In [18]:
import numpy as np

np.random.seed(42)
count_preview = 0
for input_size in hparams_space["input_size"]:
    for bs in hparams_space["batch_size"]:
        for lr in hparams_space["lr"]:
            for optimizer_name in hparams_space["optimizer"]:
                for HFlip in hparams_space["HFlip"]:
                    for VFlip in hparams_space["VFlip"]:
                        for RBContrast in hparams_space["RBContrast"]:
                            for dropout in hparams_space["dropout"]:
                                if np.random.rand() < 0.05:
                                    count_preview += 1

print(f"Se van a entrenar {count_preview} modelos con esta configuración.")

Se van a entrenar 26 modelos con esta configuración.


In [19]:
np.random.seed(42)

modelnbr = 0
for input_size in hparams_space["input_size"]:
    for bs in hparams_space["batch_size"]:
        for lr in hparams_space["lr"]:
            for optimizer_name in hparams_space["optimizer"]:
                for HFlip in hparams_space["HFlip"]:
                    for VFlip in hparams_space["VFlip"]:
                        for RBContrast in hparams_space["RBContrast"]:
                            for dropout in hparams_space["dropout"]:
                                if np.random.rand() < 0.05:
                                    modelnbr += 1
                                    print(f"\n=== Modelo #{modelnbr} | input={input_size} bs={bs} lr={lr} opt={optimizer_name} dropout={dropout} ===")

                                    hparams = {
                                        "model":       "CNNClassifier",
                                        "input_size":  input_size,
                                        "batch_size":  bs,
                                        "lr":          lr,
                                        "epochs":      hparams_space["epochs"],
                                        "optimizer":   optimizer_name,
                                        "HFlip":       HFlip,
                                        "VFlip":       VFlip,
                                        "RBContrast":  RBContrast,
                                        "loss_fn":     "CrossEntropyLoss",
                                        "train_dir":   train_dir,
                                        "val_dir":     val_dir,
                                        "es_patience": 5,
                                        "dropout":     dropout,
                                    }

                                    train_transform = A.Compose([
                                        A.Resize(hparams["input_size"], hparams["input_size"]),
                                        A.HorizontalFlip(p=hparams["HFlip"]),
                                        A.VerticalFlip(p=hparams["VFlip"]),
                                        A.RandomBrightnessContrast(p=hparams["RBContrast"]),
                                        A.Normalize(),
                                        ToTensorV2()
                                    ])
                                    val_test_transform = A.Compose([
                                        A.Resize(hparams["input_size"], hparams["input_size"]),
                                        A.Normalize(),
                                        ToTensorV2()
                                    ])

                                    train_dataset = CustomImageDataset(train_dir, transform=train_transform)
                                    val_dataset   = CustomImageDataset(val_dir,   transform=val_test_transform)
                                    t_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True)
                                    v_loader = DataLoader(val_dataset,   batch_size=bs)

                                    device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
                                    num_classes = len(set(train_dataset.labels))
                                    model = CNNClassifier(
                                        num_classes=num_classes,
                                        input_size=hparams["input_size"],
                                        dropout=hparams["dropout"]
                                    ).to(device)

                                    criterion = nn.CrossEntropyLoss()
                                    if optimizer_name == "Adam":
                                        optimizer = optim.Adam(model.parameters(), lr=hparams["lr"])
                                    else:
                                        optimizer = optim.SGD(model.parameters(), lr=hparams["lr"], momentum=0.9)

                                    hparams["count_params"] = count_parameters(model)

                                    with mlflow.start_run():
                                        mlflow.log_params(hparams)

                                        best_val_acc    = 0
                                        best_val_loss   = 0
                                        best_train_acc  = 0
                                        best_train_loss = 0
                                        best_epoch      = 0

                                        for epoch in range(hparams["epochs"]):
                                            model.train()
                                            running_loss   = 0.0
                                            correct, total = 0, 0

                                            for images, labels in t_loader:
                                                images, labels = images.to(device), labels.to(device)
                                                optimizer.zero_grad()
                                                outputs = model(images)
                                                loss    = criterion(outputs, labels)
                                                loss.backward()
                                                optimizer.step()
                                                running_loss += loss.item()
                                                _, preds = torch.max(outputs, 1)
                                                correct  += (preds == labels).sum().item()
                                                total    += labels.size(0)

                                            train_loss = running_loss / len(t_loader)
                                            train_acc  = 100.0 * correct / total
                                            val_loss, val_acc = evaluate(
                                                model, v_loader, writer, device,
                                                train_dataset.label_encoder.classes_,
                                                criterion,
                                                epoch=epoch,
                                                prefix="val"
                                            )

                                            print(f"  Epoch {epoch+1:3d}/{hparams['epochs']} | "
                                                  f"train_loss={train_loss:.4f} train_acc={train_acc:.1f}% | "
                                                  f"val_loss={val_loss:.4f} val_acc={val_acc:.1f}%")

                                            writer.add_scalar("train/loss",     train_loss, epoch)
                                            writer.add_scalar("train/accuracy", train_acc,  epoch)

                                            mlflow.log_metrics({
                                                "train_loss":     train_loss,
                                                "train_accuracy": train_acc,
                                                "val_loss":       val_loss,
                                                "val_accuracy":   val_acc
                                            }, step=epoch)

                                            if val_acc > best_val_acc:
                                                best_val_acc    = val_acc
                                                best_val_loss   = val_loss
                                                best_train_acc  = train_acc
                                                best_train_loss = train_loss
                                                best_epoch      = epoch
                                                torch.save(model.state_dict(), "cnn_model.pth")
                                                mlflow.log_artifact("cnn_model.pth")
                                            elif epoch > best_epoch + hparams["es_patience"]:
                                                print(f"  Early stopping en epoch {epoch+1}")
                                                break

                                        mlflow.log_metrics({
                                            "best_train_loss":     best_train_loss,
                                            "best_train_accuracy": best_train_acc,
                                            "best_val_loss":       best_val_loss,
                                            "best_val_accuracy":   best_val_acc,
                                            "best_epoch":          best_epoch
                                        }, step=epoch + 1)

print(f"\nBúsqueda finalizada. Se entrenaron {modelnbr} modelos.")


=== Modelo #1 | input=32 bs=16 lr=0.01 opt=Adam dropout=0.0 ===
  Epoch   1/50 | train_loss=8.9135 train_acc=23.2% | val_loss=1.9342 val_acc=30.9%
  Epoch   2/50 | train_loss=1.8669 train_acc=27.0% | val_loss=1.7509 val_acc=28.2%
  Epoch   3/50 | train_loss=1.6613 train_acc=34.1% | val_loss=1.6683 val_acc=32.0%
  Epoch   4/50 | train_loss=1.6304 train_acc=37.2% | val_loss=1.5283 val_acc=39.2%
  Epoch   5/50 | train_loss=1.6359 train_acc=31.6% | val_loss=1.5122 val_acc=39.8%
  Epoch   6/50 | train_loss=1.5608 train_acc=38.5% | val_loss=1.5961 val_acc=35.9%
  Epoch   7/50 | train_loss=1.4842 train_acc=42.0% | val_loss=1.5839 val_acc=40.3%
  Epoch   8/50 | train_loss=1.5192 train_acc=40.6% | val_loss=1.5815 val_acc=39.2%
  Epoch   9/50 | train_loss=1.4258 train_acc=43.0% | val_loss=1.4799 val_acc=45.3%
  Epoch  10/50 | train_loss=1.3663 train_acc=45.8% | val_loss=1.3652 val_acc=47.5%
  Epoch  11/50 | train_loss=1.3050 train_acc=50.8% | val_loss=1.2907 val_acc=54.7%
  Epoch  12/50 | train

In [20]:
import pandas as pd

mlflow.set_tracking_uri("sqlite:///mlflow_3.db")
runs = mlflow.search_runs(
    experiment_names=["Clasificador_Imagenes"],
    order_by=["metrics.best_val_accuracy DESC"]
)

cols = ["params.model", "params.input_size", "params.batch_size", "params.lr",
        "params.optimizer", "params.dropout", "params.HFlip", "params.VFlip",
        "params.RBContrast", "metrics.best_val_accuracy", "metrics.best_train_accuracy",
        "metrics.best_epoch"]
cols = [c for c in cols if c in runs.columns]

print(f"Total de runs CNN: {len(runs)}")
runs[cols].head(10)

Total de runs CNN: 26


,params.model,params.input_size,params.batch_size,params.lr,params.optimizer,params.dropout,params.HFlip,params.VFlip,params.RBContrast,metrics.best_val_accuracy,metrics.best_train_accuracy,metrics.best_epoch
0,CNNClassifier,64,16,0.001,Adam,0.2,0.5,0.5,0.0,75.138122,73.170732,17.0
1,CNNClassifier,64,64,0.001,Adam,0.0,0.5,0.5,0.0,74.033149,88.522238,21.0
2,CNNClassifier,64,64,0.0001,Adam,0.0,0.0,0.5,0.0,72.375691,92.395983,31.0
3,CNNClassifier,64,16,0.001,SGD,0.0,0.0,0.5,0.0,71.823204,86.083214,15.0
4,CNNClassifier,32,16,0.001,Adam,0.0,0.5,0.0,0.5,71.823204,79.483501,12.0
5,CNNClassifier,32,64,0.0001,Adam,0.2,0.5,0.0,0.5,70.718232,81.061693,24.0
6,CNNClassifier,32,64,0.001,Adam,0.0,0.0,0.0,0.0,70.718232,100.000000,17.0
7,CNNClassifier,32,16,0.0001,Adam,0.0,0.5,0.0,0.0,70.718232,98.134864,22.0
8,CNNClassifier,64,16,0.01,SGD,0.0,0.0,0.0,0.0,70.165746,86.800574,17.0
9,CNNClassifier,32,64,0.0001,Adam,0.0,0.5,0.0,0.0,69.613260,98.995696,34.0
